# 05 — Locked holdout & business report

Notebook duy nhất được score LFW fold 9 và PAD subject holdout. Notebook không import project code; nó chỉ nhận hai **locked DS artifacts** từ notebook 03–04. Việc phụ thuộc artifact là chủ ý để bảo vệ holdout, không phải phụ thuộc phần ML Engineer.

Workflow an toàn: chạy các cell kiểm tra lock trước, đọc bảng validation, rồi chỉ đổi `UNLOCK_HOLDOUT = True` khi chủ ý thực hiện đúng một lần. Không đổi backbone, epoch hoặc threshold sau khi xem holdout.


In [1]:
import os
import sys
import json
import random
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "scikit-learn>=1.5", "pandas>=2.2", "seaborn>=0.13",
    "opencv-python-headless>=4.10", "tqdm>=4.66",
])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter

SEED = 464
random.seed(SEED)
np.random.seed(SEED)

# A blank Colab runtime is the reference environment.  Nothing from src/ or
# scripts/ is needed.  All intermediate DS assets live under this runtime root.
RUNTIME_ROOT = Path("/content/facekyc_ds") if Path("/content").exists() else Path.cwd() / ".facekyc_ds"
DATA_ROOT = RUNTIME_ROOT / "data"
REPORT_ROOT = RUNTIME_ROOT / "reports"
ARTIFACT_ROOT = RUNTIME_ROOT / "artifacts"
for directory in (DATA_ROOT, REPORT_ROOT, ARTIFACT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print("Runtime root:", RUNTIME_ROOT)
print("Python:", sys.version.split()[0])


Runtime root: /content/facekyc_ds
Python: 3.13.15


In [2]:
from sklearn.datasets import fetch_lfw_people

# This call downloads the official funneled LFW archive plus pair protocols.
_ = fetch_lfw_people(
    data_home=str(DATA_ROOT), color=True, resize=0.5,
    min_faces_per_person=0, download_if_missing=True,
)
LFW_HOME = DATA_ROOT / "lfw_home"
LFW_IMAGE_ROOT = LFW_HOME / "lfw_funneled"
assert LFW_IMAGE_ROOT.exists(), f"LFW extraction failed: {LFW_IMAGE_ROOT}"

def subject_bucket(subject):
    # Stable subject-level split: 80% train, 10% validation, 10% locked holdout.
    return int(hashlib.sha256(subject.encode("utf-8")).hexdigest()[:8], 16) % 10

def build_lfw_manifest():
    rows = []
    for path in sorted(LFW_IMAGE_ROOT.glob("*/*.jpg")):
        subject = path.parent.name
        bucket = subject_bucket(subject)
        split = "train" if bucket < 8 else ("validation" if bucket == 8 else "holdout")
        rows.append({"path": str(path), "subject": subject, "split": split})
    frame = pd.DataFrame(rows)
    assert len(frame) == 13233, f"Expected 13,233 LFW images, found {len(frame)}"
    assert frame.groupby("subject")["split"].nunique().max() == 1
    return frame

MANIFEST_PATH = DATA_ROOT / "lfw_subject_manifest.csv"
manifest = build_lfw_manifest()
manifest.to_csv(MANIFEST_PATH, index=False)
print("LFW images:", len(manifest), "subjects:", manifest.subject.nunique())


LFW images: 13233 subjects: 5749


In [3]:
from sklearn.metrics import roc_auc_score

def verification_metrics(labels, scores, threshold):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    accepted = scores >= threshold
    impostor = labels == 0
    genuine = labels == 1
    return {
        "threshold": float(threshold),
        "fmr": float(accepted[impostor].mean()),
        "fnmr": float((~accepted[genuine]).mean()),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, scores)),
        "pairs": int(len(labels)),
    }

def threshold_for_fmr(labels, scores, target_fmr=0.01):
    labels = np.asarray(labels).astype(int)
    scores = np.asarray(scores, dtype=float)
    impostor_scores = np.sort(scores[labels == 0])
    rank = int(np.ceil((1.0 - target_fmr) * len(impostor_scores))) - 1
    rank = int(np.clip(rank, 0, len(impostor_scores) - 1))
    return float(np.nextafter(impostor_scores[rank], np.inf))

def pad_metrics(labels, live_scores, threshold):
    labels = np.asarray(labels).astype(int)  # 0=attack, 1=bona fide
    live_scores = np.asarray(live_scores, dtype=float)
    accepted = live_scores >= threshold
    attack = labels == 0
    bona_fide = labels == 1
    apcer = float(accepted[attack].mean())
    bpcer = float((~accepted[bona_fide]).mean())
    return {
        "threshold": float(threshold), "apcer": apcer, "bpcer": bpcer,
        "acer": float((apcer + bpcer) / 2),
        "accuracy": float((accepted == labels).mean()),
        "auc": float(roc_auc_score(labels, live_scores)),
        "samples": int(len(labels)),
    }

def threshold_for_apcer(labels, live_scores, target_apcer=0.05):
    labels = np.asarray(labels).astype(int)
    attack_scores = np.sort(np.asarray(live_scores, dtype=float)[labels == 0])
    rank = int(np.ceil((1.0 - target_apcer) * len(attack_scores))) - 1
    rank = int(np.clip(rank, 0, len(attack_scores) - 1))
    return float(np.nextafter(attack_scores[rank], np.inf))


In [4]:
def pair_image_path(subject, one_based_index):
    return LFW_IMAGE_ROOT / subject / f"{subject}_{int(one_based_index):04d}.jpg"

def parse_pair_line(line):
    parts = line.strip().split("\t")
    if len(parts) == 3:
        return pair_image_path(parts[0], parts[1]), pair_image_path(parts[0], parts[2]), 1
    if len(parts) == 4:
        return pair_image_path(parts[0], parts[1]), pair_image_path(parts[2], parts[3]), 0
    raise ValueError(f"Malformed LFW pair line: {line!r}")

def load_official_fold_records(folds):
    # Parse only the folds explicitly requested by the gated caller below.
    protocol_path = LFW_HOME / "pairs.txt"
    with protocol_path.open(encoding="utf-8") as handle:
        header = next(handle).strip()
        assert header == "10\t300", header
        requested = set(folds)
        records = []
        for row_index, line in enumerate(handle):
            fold = row_index // 600
            if fold not in requested:
                continue
            left, right, label = parse_pair_line(line)
            assert left.exists() and right.exists()
            records.append({"left": str(left), "right": str(right), "label": label, "fold": fold})
    return pd.DataFrame(records)


In [5]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch==2.6.0"
])
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

class PairDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        row = self.records.iloc[index]
        with Image.open(row.left) as image:
            left = self.transform(image.convert("RGB"))
        with Image.open(row.right) as image:
            right = self.transform(image.convert("RGB"))
        return left, right, int(row.label), int(row.fold)

def build_backbone(name):
    if name not in {"vggface2", "casia-webface"}:
        raise ValueError(name)
    model = InceptionResnetV1(pretrained=name, classify=False).eval().to(DEVICE)

    def transform(image):
        # fixed_image_standardization expects float pixels in [0, 255].
        # torchvision.transforms.ToTensor() would first scale them to [0, 1]
        # and collapse FaceNet embeddings to an almost constant vector.
        resized = image.resize((160, 160), Image.Resampling.BILINEAR)
        pixels = np.asarray(resized, dtype=np.float32)
        tensor = torch.from_numpy(pixels).permute(2, 0, 1)
        return fixed_image_standardization(tensor)

    return model, transform

@torch.inference_mode()
def score_pairs(model_name, records, batch_size=64):
    model, transform = build_backbone(model_name)
    loader = DataLoader(PairDataset(records, transform), batch_size=batch_size,
                        shuffle=False, num_workers=2, pin_memory=DEVICE.type == "cuda")
    scores, labels, folds = [], [], []
    for left, right, label, fold in loader:
        left = F.normalize(model(left.to(DEVICE, non_blocking=True)), dim=1)
        right = F.normalize(model(right.to(DEVICE, non_blocking=True)), dim=1)
        scores.extend((left * right).sum(dim=1).cpu().numpy().tolist())
        labels.extend(label.numpy().tolist())
        folds.extend(fold.numpy().tolist())
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return np.asarray(scores), np.asarray(labels), np.asarray(folds)


Device: cuda Tesla T4


In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

INPUT_SIZE = 224
TARGET_APCER = 0.05
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def synthetic_presentation_attack(image, seed):
    """Deterministic print/replay proxy; not a real PAD capture."""
    rng = np.random.default_rng(seed)
    image = image.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR).convert("RGB")
    array = np.asarray(image).astype(np.float32)
    attack_type = int(seed % 3)
    if attack_type == 0:  # print: paper tint, reduced gamut, dot/noise texture
        gray = array.mean(axis=2, keepdims=True)
        array = 0.65 * array + 0.35 * gray
        array *= np.array([1.04, 1.00, 0.90], dtype=np.float32)
        array += rng.normal(0, 7, array.shape)
        array[::4, ::4] *= 0.82
    elif attack_type == 1:  # replay: scanlines, blue cast and screen glare
        array *= np.array([0.92, 0.98, 1.10], dtype=np.float32)
        array[::3] *= 0.72
        x = np.linspace(-1, 1, array.shape[1])[None, :, None]
        glare = 34 * np.exp(-((x - 0.35) ** 2) / 0.05)
        array += glare
    else:  # recapture: resampling, compression-like blocking and moire
        small = Image.fromarray(np.clip(array, 0, 255).astype(np.uint8)).resize((64, 64))
        array = np.asarray(small.resize((INPUT_SIZE, INPUT_SIZE), Image.Resampling.BILINEAR)).astype(np.float32)
        yy, xx = np.indices(array.shape[:2])
        moire = 12 * np.sin((xx + yy) * 0.42)[..., None]
        array += moire
    return Image.fromarray(np.clip(array, 0, 255).astype(np.uint8))

class PADProxyDataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        augment = [transforms.Resize((INPUT_SIZE, INPUT_SIZE))]
        if training:
            augment += [transforms.RandomHorizontalFlip(), transforms.ColorJitter(0.12, 0.12, 0.08, 0.02)]
        augment += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
        self.transform = transforms.Compose(augment)

    def __len__(self):
        return len(self.frame) * 2

    def __getitem__(self, index):
        base_index, label = divmod(index, 2)  # 0=attack, 1=bona fide
        row = self.frame.iloc[base_index]
        with Image.open(row.path) as source:
            image = source.convert("RGB")
        if label == 0:
            seed = int(hashlib.sha256(row.path.encode()).hexdigest()[:8], 16)
            image = synthetic_presentation_attack(image, seed)
        return self.transform(image), torch.tensor(label, dtype=torch.long), str(row.subject)

def build_pad_cnn(name):
    if name == "mobilenet_v3_small":
        model = models.mobilenet_v3_small(weights=None)
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
    elif name == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 2)
    else:
        raise ValueError(name)
    return model.to(DEVICE)

def evaluate_pad(model, loader):
    model.eval()
    scores, labels = [], []
    with torch.inference_mode():
        for images, target, _ in loader:
            probabilities = torch.softmax(model(images.to(DEVICE, non_blocking=True)), dim=1)[:, 1]
            scores.extend(probabilities.cpu().numpy().tolist())
            labels.extend(target.numpy().astype(int).tolist())
    return np.asarray(labels), np.asarray(scores)


Device: cuda Tesla T4


In [7]:
verification_report_path = REPORT_ROOT / "03_verification_selection.json"
pad_report_path = REPORT_ROOT / "04_pad_selection.json"
pad_checkpoint_path = ARTIFACT_ROOT / "pad_proxy_selected.pt"
for required in [verification_report_path, pad_report_path, pad_checkpoint_path]:
    assert required.exists(), f"Run notebooks 03 and 04 first; missing {required}"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

verification_lock = json.loads(verification_report_path.read_text(encoding="utf-8"))
pad_lock = json.loads(pad_report_path.read_text(encoding="utf-8"))
assert verification_lock["holdout_accessed"] is False
assert verification_lock["selection_folds"] == list(range(8))
assert verification_lock["validation_fold"] == 8
assert verification_lock["holdout_fold"] == 9
assert verification_lock["selected_backbone"] in {"vggface2", "casia-webface"}
assert np.isfinite(float(verification_lock["locked_threshold"]))

assert pad_lock["holdout_accessed"] is False
assert pad_lock["production_ready"] is False
assert pad_lock["subject_disjoint"] is True
assert pad_lock["input_size"] == INPUT_SIZE
assert pad_lock["target_apcer"] == TARGET_APCER
assert pad_lock["score"] == "softmax_probability_class_1"
assert sha256_file(pad_checkpoint_path) == pad_lock["checkpoint_sha256"]
checkpoint = torch.load(pad_checkpoint_path, map_location="cpu", weights_only=True)
assert checkpoint["architecture"] == pad_lock["selected_architecture"]
assert checkpoint["input_size"] == pad_lock["input_size"]
assert checkpoint["class_contract"] == {"0": "attack", "1": "bona_fide"}
assert checkpoint["score"] == pad_lock["score"]
assert np.isclose(float(checkpoint["threshold"]), float(pad_lock["locked_threshold"]))

lock_summary = pd.DataFrame([
    {"stage": "face_verification", "model": verification_lock["selected_backbone"],
     "threshold": verification_lock["locked_threshold"], **verification_lock["validation_metrics"]},
    {"stage": "pad_proxy", "model": pad_lock["selected_architecture"],
     "threshold": pad_lock["locked_threshold"], **pad_lock["validation_metrics"]},
])
display(lock_summary)

HOLDOUT_REPORT = REPORT_ROOT / "05_locked_holdout.json"
assert not HOLDOUT_REPORT.exists(), "Locked holdout was already evaluated in this runtime."
UNLOCK_HOLDOUT = True  # Review lock_summary, then change to True for one deliberate run.
print("Locks verified. Holdout remains locked:", not UNLOCK_HOLDOUT)


,stage,model,threshold,fmr,fnmr,accuracy,auc,pairs,apcer,bpcer,acer,samples
0,face_verification,vggface2,0.422526,0.01,0.046667,0.971667,0.993622,600.0,NaN,NaN,NaN,NaN
1,pad_proxy,mobilenet_v3_small,0.761909,NaN,NaN,0.975000,0.999081,NaN,0.05,0.0,0.025,800.0


Locks verified. Holdout remains locked: False


In [8]:
if not UNLOCK_HOLDOUT:
    verification_holdout = None
    print("SKIPPED: holdout vẫn khóa. Review lock_summary, đổi UNLOCK_HOLDOUT=True rồi chạy lại cell này.")
else:
    assert not HOLDOUT_REPORT.exists(), "Locked holdout was already evaluated in this runtime."
    assert DEVICE.type == "cuda", "Notebook 05 nên chạy bằng Colab GPU."
    holdout_pairs = load_official_fold_records([9])
    assert len(holdout_pairs) == 600
    assert holdout_pairs.groupby("label").size().to_dict() == {0: 300, 1: 300}
    verification_scores, verification_labels, verification_folds = score_pairs(
        verification_lock["selected_backbone"], holdout_pairs
    )
    assert set(verification_folds) == {9}
    assert set(verification_labels) == {0, 1}
    assert np.isfinite(verification_scores).all()
    verification_holdout = verification_metrics(
        verification_labels, verification_scores, verification_lock["locked_threshold"]
    )
    display(verification_holdout)


{'threshold': 0.4225261211395264,
 'fmr': 0.006666666666666667,
 'fnmr': 0.03666666666666667,
 'accuracy': 0.9783333333333334,
 'auc': 0.997411111111111,
 'pairs': 600}

## PAD subject holdout — fixed CNN and threshold


In [9]:
if not UNLOCK_HOLDOUT:
    pad_holdout = None
    print("SKIPPED: holdout vẫn khóa.")
elif verification_holdout is None:
    pad_holdout = None
    print("SKIPPED: hãy chạy face-verification holdout trước.")
else:
    assert not HOLDOUT_REPORT.exists(), "Locked holdout was already evaluated in this runtime."
    pad_model = build_pad_cnn(checkpoint["architecture"])
    pad_model.load_state_dict(checkpoint["state_dict"])
    pad_model.eval()
    holdout_frame = manifest.query("split == 'holdout'")
    development_subjects = set(manifest.query("split != 'holdout'").subject)
    holdout_subjects = set(holdout_frame.subject)
    assert holdout_subjects and not (development_subjects & holdout_subjects)
    holdout_loader = DataLoader(PADProxyDataset(holdout_frame, training=False), batch_size=128,
                                shuffle=False, num_workers=2, pin_memory=DEVICE.type == "cuda")
    pad_labels, pad_scores = evaluate_pad(pad_model, holdout_loader)
    assert set(pad_labels) == {0, 1}
    assert np.isfinite(pad_scores).all()
    pad_holdout = pad_metrics(pad_labels, pad_scores, pad_lock["locked_threshold"])
    del pad_model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    display(pad_holdout)


{'threshold': 0.7619086503982545,
 'apcer': 0.05776892430278884,
 'bpcer': 0.00099601593625498,
 'acer': 0.02938247011952191,
 'accuracy': 0.9706175298804781,
 'auc': 0.9987103379311439,
 'samples': 2008}

In [10]:
if not UNLOCK_HOLDOUT:
    final_report = None
    print("SKIPPED: holdout vẫn khóa; chưa tạo final report.")
elif verification_holdout is None or pad_holdout is None:
    final_report = None
    print("SKIPPED: cần hoàn tất cả hai holdout cell trước.")
else:
    assert not HOLDOUT_REPORT.exists(), "Locked holdout was already evaluated in this runtime."
    screening_gates = {
        "verification_fmr_le_0_02": verification_holdout["fmr"] <= 0.02,
        "verification_fnmr_le_0_20": verification_holdout["fnmr"] <= 0.20,
        "pad_proxy_apcer_le_0_10": pad_holdout["apcer"] <= 0.10,
        "pad_proxy_bpcer_le_0_20": pad_holdout["bpcer"] <= 0.20,
    }
    screening_passed = all(screening_gates.values())
    final_report = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "verification": {
            "backbone": verification_lock["selected_backbone"],
            "validation": verification_lock["validation_metrics"],
            "locked_holdout": verification_holdout,
        },
        "pad_proxy": {
            "architecture": pad_lock["selected_architecture"],
            "validation": pad_lock["validation_metrics"],
            "locked_holdout": pad_holdout,
        },
        "deployment_status": "candidate_research_only",
        "holdout_accessed": True,
        "holdout_evaluation_count": 1,
        "screening_gates": screening_gates,
        "screening_passed": screening_passed,
        "artifact_integrity": {"pad_checkpoint_sha256": pad_lock["checkpoint_sha256"]},
        "decision": (
            "Proceed to MLE packaging as a research candidate only; production approval remains fail-closed pending target-domain verification and real PAD evaluation."
            if screening_passed else
            "Stop research-candidate packaging and investigate failed locked-holdout screening gates without retuning on holdout."
        ),
        "limitations": [
            "LFW is not an ID-document/selfie benchmark for Vietnamese eKYC.",
            "The ImageNet-pretrained PAD CNN was fine-tuned only on synthetic LFW-derived proxy attacks.",
            "Synthetic PAD attacks do not measure resistance to real print, replay, mask or deepfake attacks.",
            "No representative subgroup fairness audit or ISO/IEC 30107-3 certification.",
        ],
    }
    HOLDOUT_REPORT.write_text(json.dumps(final_report, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    scorecard = pd.DataFrame([
        {"stage": "face_verification", **verification_holdout},
        {"stage": "pad_proxy", **pad_holdout},
    ])
    display(scorecard)
    display(final_report)


,stage,threshold,fmr,fnmr,accuracy,auc,pairs,apcer,bpcer,acer,samples
0,face_verification,0.422526,0.006667,0.036667,0.978333,0.997411,600.0,NaN,NaN,NaN,NaN
1,pad_proxy,0.761909,NaN,NaN,0.970618,0.998710,NaN,0.057769,0.000996,0.029382,2008.0


{'created_at': '2026-08-31T04:59:25.041451+00:00',
 'verification': {'backbone': 'vggface2',
  'validation': {'threshold': 0.4225261211395264,
   'fmr': 0.01,
   'fnmr': 0.04666666666666667,
   'accuracy': 0.9716666666666667,
   'auc': 0.9936222222222222,
   'pairs': 600},
  'locked_holdout': {'threshold': 0.4225261211395264,
   'fmr': 0.006666666666666667,
   'fnmr': 0.03666666666666667,
   'accuracy': 0.9783333333333334,
   'auc': 0.997411111111111,
   'pairs': 600}},
 'pad_proxy': {'architecture': 'mobilenet_v3_small',
  'validation': {'threshold': 0.7619086503982545,
   'apcer': 0.05,
   'bpcer': 0.0,
   'acer': 0.025,
   'accuracy': 0.975,
   'auc': 0.99908125,
   'samples': 800},
  'locked_holdout': {'threshold': 0.7619086503982545,
   'apcer': 0.05776892430278884,
   'bpcer': 0.00099601593625498,
   'acer': 0.02938247011952191,
   'accuracy': 0.9706175298804781,
   'auc': 0.9987103379311439,
   'samples': 2008}},
 'deployment_status': 'candidate_research_only',
 'holdout_accesse

## Business interpretation

- `verified`: similarity và PAD proxy vượt ngưỡng, ảnh qua quality gates; vẫn cần các kiểm soát KYC khác.
- `manual_review`: score sát ngưỡng hoặc có cảnh báo mềm; không tự động chuyển thành reject.
- `recapture`: no-face, multi-face, confidence thấp hoặc ảnh hỏng.
- `not_verified`: similarity dưới ngưỡng hoặc PAD score dưới ngưỡng.

Sau khi reviewer chấp nhận báo cáo này mới được trích logic đã kiểm chứng sang `src/`, định nghĩa artifact contract, API, monitoring và deployment. Trạng thái production vẫn fail-closed vì PAD mới là synthetic proxy. File `05_locked_holdout.json` là immutable evidence trong runtime: nếu đã tồn tại thì không xóa để chạy lại hoặc retune theo holdout.
